In [2]:
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    Gemma3ForConditionalGeneration
)

In [7]:
data_file = "../AI/dataset/tool_just_for_train_check.jsonl"
student_model_path = "../AI/models/functiongemma-270m-it-v2"
teacher_model_path = "../AI/models/gemma-3-4b-it"
nepoch = 1

In [4]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


In [5]:
teacher_tokenizer = AutoTokenizer.from_pretrained(teacher_model_path)

teacher_model = Gemma3ForConditionalGeneration.from_pretrained(
    teacher_model_path,
    torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
    low_cpu_mem_usage=True
).eval()

teacher_model.to(device)
print("Teacher model device:", next(teacher_model.parameters()).device)

Loading weights: 100%|██████████| 883/883 [00:00<00:00, 1995.96it/s]


Teacher model device: cuda:0


In [10]:
student_tokenizer = AutoTokenizer.from_pretrained(student_model_path)
student_model = AutoModelForCausalLM.from_pretrained(
    student_model_path,
    torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
)
student_model.to(device)
print("Student model device:", next(student_model.parameters()).device)

Loading weights: 100%|██████████| 236/236 [00:00<00:00, 3039.15it/s]


Student model device: cuda:0


In [11]:
print("Student details:")
print("Input embeddings shape:", student_model.get_input_embeddings().weight.shape)
print("Output embeddings shape:", student_model.get_output_embeddings().weight.shape)
print("Tokenizer length:", len(student_tokenizer))

Student details:
Input embeddings shape: torch.Size([262144, 640])
Output embeddings shape: torch.Size([262144, 640])
Tokenizer length: 262146


In [12]:
print("Teacher details:")
print("Input embeddings shape:", teacher_model.get_input_embeddings().weight.shape)
print("Output embeddings shape:", teacher_model.get_output_embeddings().weight.shape)
print("Tokenizer length:", len(teacher_tokenizer))

Teacher details:
Input embeddings shape: torch.Size([262208, 2560])
Output embeddings shape: torch.Size([262208, 2560])
Tokenizer length: 262145


In [13]:
teacher_vocab_size = teacher_model.get_output_embeddings().weight.shape[0]
student_model.resize_token_embeddings(teacher_vocab_size)

[transformers] The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`


Gemma3TextScaledWordEmbedding(262208, 640, padding_idx=0)

In [14]:
print("Student details:")
print("Input embeddings shape:", student_model.get_input_embeddings().weight.shape)
print("Output embeddings shape:", student_model.get_output_embeddings().weight.shape)
print("Tokenizer length:", len(student_tokenizer))

print("Teacher details:")
print("Input embeddings shape:", teacher_model.get_input_embeddings().weight.shape)
print("Output embeddings shape:", teacher_model.get_output_embeddings().weight.shape)
print("Tokenizer length:", len(teacher_tokenizer))

Student details:
Input embeddings shape: torch.Size([262208, 640])
Output embeddings shape: torch.Size([262208, 640])
Tokenizer length: 262146
Teacher details:
Input embeddings shape: torch.Size([262208, 2560])
Output embeddings shape: torch.Size([262208, 2560])
Tokenizer length: 262145


In [16]:
def build_prompt_and_full(messages, tools):
    prompt = student_tokenizer.apply_chat_template(
        messages[:-1],
        tools=tools,
        tokenize=False,
        add_generation_prompt=True,
    )
    full = student_tokenizer.apply_chat_template(
        messages,
        tools=tools,
        tokenize=False,
        add_generation_prompt=False,
    )
    prompt_ids = student_tokenizer(prompt, return_tensors="pt").input_ids[0]
    return prompt, full, prompt_ids.shape[0]

In [17]:
def prepare_example(example):
    prompt, full, prompt_len = build_prompt_and_full(example["messages"], example["tools"])
    return {
        "prompt": prompt,
        "full": full,
        "prompt_len": prompt_len,
        "split": example.get("metadata", "train"),
    }


In [18]:
raw_dataset = load_dataset("json", data_files=data_file, split="train")
processed = raw_dataset.map(prepare_example, remove_columns=raw_dataset.column_names)

train_dataset = processed.filter(lambda ex: ex["split"] == "train")
eval_dataset = processed.filter(lambda ex: ex["split"] == "test")

Generating train split: 4 examples [00:00, 314.16 examples/s]
Filter: 100%|██████████| 4/4 [00:00<00:00, 1875.81 examples/s]


In [19]:
print(train_dataset[0])

{'prompt': "<bos><start_of_turn>developer\nYou are an on-device smart home controller. Given a natural language command from the user, call the appropriate smart home function. If the user does not specify a required value (e.g. which room or what temperature), omit that parameter from the function call. Maintain context across conversation turns to resolve pronouns and sequential commands.<start_function_declaration>declaration:toggle_lights{description:<escape>Turn lights on or off in a specified room.<escape>,parameters:{properties:{room:{description:<escape>The room whose lights to control.<escape>,enum:[<escape>living_room<escape>,<escape>bedroom<escape>,<escape>kitchen<escape>,<escape>bathroom<escape>,<escape>office<escape>,<escape>hallway<escape>],type:<escape>STRING<escape>},state:{description:<escape>Whether to turn lights on or off.<escape>,enum:[<escape>on<escape>,<escape>off<escape>],type:<escape>STRING<escape>}},type:<escape>OBJECT<escape>}}<end_function_declaration><start

In [20]:
def collate_fn(batch):
    full_texts = [example["full"] for example in batch]
    prompt_lens = torch.tensor([example["prompt_len"] for example in batch], dtype=torch.long)

    tokenized = student_tokenizer(
        full_texts,
        padding=True,
        truncation=True,
        return_tensors="pt",
    )

    labels = tokenized.input_ids.clone()
    for i, prompt_len in enumerate(prompt_lens):
        labels[i, :prompt_len] = -100

    return {
        "input_ids": tokenized.input_ids,
        "attention_mask": tokenized.attention_mask,
        "labels": labels,
        "prompt_lens": prompt_lens,
        "full_texts": full_texts,
    }


In [21]:
train_loader = DataLoader(
    train_dataset,
    batch_size=1,          # try 8, 16, or higher
    shuffle=True,
    collate_fn=collate_fn,
    num_workers=0,         # parallel CPU workers
    pin_memory=True        # faster transfer to GPU
)
optimizer = torch.optim.AdamW(student_model.parameters(), lr=1e-5)

alpha = 0.5   # KD weight
T = 2.0       # temperature for distillation
student_model.train()

Gemma3ForCausalLM(
  (model): Gemma3TextModel(
    (embed_tokens): Gemma3TextScaledWordEmbedding(262208, 640, padding_idx=0)
    (layers): ModuleList(
      (0-17): 18 x Gemma3DecoderLayer(
        (self_attn): Gemma3Attention(
          (q_proj): Linear(in_features=640, out_features=1024, bias=False)
          (k_proj): Linear(in_features=640, out_features=256, bias=False)
          (v_proj): Linear(in_features=640, out_features=256, bias=False)
          (o_proj): Linear(in_features=1024, out_features=640, bias=False)
          (q_norm): Gemma3RMSNorm((256,), eps=1e-06)
          (k_norm): Gemma3RMSNorm((256,), eps=1e-06)
        )
        (mlp): Gemma3MLP(
          (gate_proj): Linear(in_features=640, out_features=2048, bias=False)
          (up_proj): Linear(in_features=640, out_features=2048, bias=False)
          (down_proj): Linear(in_features=2048, out_features=640, bias=False)
          (act_fn): GELUTanh()
        )
        (input_layernorm): Gemma3RMSNorm((640,), eps=1e-06)

In [22]:
for epoch in range(nepoch):
    for batch in train_loader:
        # Move only tensors to GPU
        batch = {
            k: (v.to(device) if isinstance(v, torch.Tensor) else v)
            for k, v in batch.items()
        }
        print(f"Batch input_ids shape: {batch['input_ids'].shape}")
        print(f"Batch labels shape: {batch['labels'].shape}")
        # ---- Teacher forward ----
        with torch.no_grad():
            teacher_inputs = teacher_tokenizer(
                batch["full_texts"],   # raw strings
                return_tensors="pt",
                padding=True,
                truncation=True,
            ).to(device)

            teacher_logits = teacher_model(**teacher_inputs).logits
        
        print(f"Teacher logits shape: {teacher_logits.shape}")

        # ---- Student forward ----
        student_logits = student_model(
            input_ids=batch["input_ids"],
            attention_mask=batch["attention_mask"],
        ).logits
        print(f"Student logits shape: {student_logits.shape}")
        
        # ---- Align sequence lengths ----
        min_len = min(student_logits.size(1), teacher_logits.size(1))
        print(f"Min length: {min_len}")
        student_logits = student_logits[:, :min_len, :]
        teacher_logits = teacher_logits[:, :min_len, :]
        labels = batch["labels"][:, :min_len]

        # ---- KD loss ----
        teacher_probs = F.softmax(teacher_logits / T, dim=-1)
        student_log_probs = F.log_softmax(student_logits / T, dim=-1)

        print("Teacher probs shape:", teacher_probs.shape)
        print("Student log probs shape:", student_log_probs.shape)

        mask = labels != -100
        kd_loss = F.kl_div(student_log_probs, teacher_probs, reduction="none").sum(-1)
        kd_loss = (kd_loss * mask).sum() / mask.sum().clamp_min(1)

        print(f"KD loss: {kd_loss.item():.4f}")
        
        # ---- CE loss ----
        ce_loss = F.cross_entropy(
            student_logits.view(-1, student_logits.size(-1)),
            labels.view(-1),
            ignore_index=-100,
        )
        print(f"CE loss: {ce_loss.item():.4f}")

        # ---- Final loss ----
        loss = alpha * kd_loss + (1 - alpha) * ce_loss

        # ---- Backward ----
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

    print(f"Epoch {epoch+1} loss = {loss.item():.4f}")


Batch input_ids shape: torch.Size([1, 816])
Batch labels shape: torch.Size([1, 816])
Teacher logits shape: torch.Size([1, 1117, 262208])
Student logits shape: torch.Size([1, 816, 262208])
Min length: 816
Teacher probs shape: torch.Size([1, 816, 262208])
Student log probs shape: torch.Size([1, 816, 262208])
KD loss: 23.1250
CE loss: 47.5000
Batch input_ids shape: torch.Size([1, 800])
Batch labels shape: torch.Size([1, 800])
Teacher logits shape: torch.Size([1, 1102, 262208])
Student logits shape: torch.Size([1, 800, 262208])
Min length: 800
Teacher probs shape: torch.Size([1, 800, 262208])
Student log probs shape: torch.Size([1, 800, 262208])
KD loss: 17.5000
CE loss: 38.7500
Epoch 1 loss = 28.1250


In [23]:
student_model.save_pretrained("models/functiongemma-270m-it-kd-3")
student_tokenizer.save_pretrained("models/functiongemma-270m-it-kd-3")

Writing model shards: 100%|██████████| 1/1 [00:05<00:00,  5.92s/it]


('models/functiongemma-270m-it-kd-3/tokenizer_config.json',
 'models/functiongemma-270m-it-kd-3/chat_template.jinja',
 'models/functiongemma-270m-it-kd-3/tokenizer.json')

In [24]:
for i in range(262208 - len(student_tokenizer)):
    student_tokenizer.add_tokens([f"<extra_token_{i}>"])

student_model.resize_token_embeddings(len(student_tokenizer))
student_model.save_pretrained("models/functiongemma-270m-it-kd-resized-3")
student_tokenizer.save_pretrained("models/functiongemma-270m-it-kd-resized-3")

Writing model shards: 100%|██████████| 1/1 [00:02<00:00,  2.79s/it]


('models/functiongemma-270m-it-kd-resized-3/tokenizer_config.json',
 'models/functiongemma-270m-it-kd-resized-3/chat_template.jinja',
 'models/functiongemma-270m-it-kd-resized-3/tokenizer.json')